# torch-cnmf Inference 

| Step | Function | Description |
|------|----------|-------------|
| **Step 1. Set Parameters** | — | Define all NMF hyperparameters (`n_iter`, `K`, `algo`, `mode`, `tol`, etc.), I/O paths, and annotation settings |
| **Step 2. Load Seeds** | `np.load` | (Optional) Load pre-computed NMF seeds for reproducible initialization |
| **Step 3. Load Data** | `sc.read` | Read the raw counts `.h5ad` file |
| **Step 4. Filter Genes** | — | (Optional) Remove non-coding genes (Ensembl-only IDs), save filtered `.h5ad` to `Inference/` |
| **Step 5. Prepare** | `cNMF()`, `cnmf_obj.prepare()` | Initialize cNMF object, normalize counts, select high-variance genes, set up factorization parameters |
| **Step 6. Factorize** | `cnmf_obj.factorize()` | Run `n_iter` NMF replicates for each K value (GPU-accelerated) |
| **Step 7. Combine** | `cnmf_obj.combine()` | Merge replicate spectra (H) and usage (W) matrices across iterations |
| **Step 8. K Selection** | `cnmf_obj.k_selection_plot()` | Plot stability vs. error for each K to guide component selection |
| **Step 9. Consensus** | `run_cnmf_consensus()` | Density-filter and cluster replicates into consensus programs for each (K, threshold) |
| **Step 10. Save Results to MuData** | `compile_results()` | Package consensus results into MuData `.h5mu` files with `rna` + `cNMF` modalities |
| **Step 11. Gene Annotation** | `annotate_genes_to_excel()` | Annotate top genes per program using Enrichr and save to Excel |
| **Step 12. Diagnostic Plots** | `generate_all_plots()` | Generate elbow curves, usage heatmaps, and loading violin plots per K × threshold |

## Expected Output Structure

After running the full inference pipeline, the output directory will have the following structure:

```
{output_directory}/{run_name}/
└── Inference/
    ├── adata_without_noncoding.h5ad                     # Filtered counts (non-coding genes removed)
    │
    ├── cnmf_tmp/                                        # Internal cNMF working files
    │   ├── Inference.norm_counts.h5ad                   # Normalized + HVG-filtered counts
    │   ├── Inference.overdispersed_genes.txt             # List of selected high-variance genes
    │   ├── Inference.k_selection_stats.df.npz            # Stability and error stats for K selection
    │   ├── Inference.nmf_idvrun_spectra.k_{K}.iter_{i}.df.npz  # Per-replicate H matrix
    │   ├── Inference.nmf_idvrun_usage.k_{K}.iter_{i}.df.npz    # Per-replicate W matrix
    │   ├── Inference.spectra.k_{K}.merged.df.npz         # Merged H across replicates
    │   ├── Inference.usages.k_{K}.merged.df.npz          # Merged W across replicates
    │   ├── Inference.spectra.k_{K}.dt_{thresh}.consensus.df.npz  # Consensus H
    │   ├── Inference.usages.k_{K}.dt_{thresh}.consensus.df.npz   # Consensus W
    │   └── Inference.clustering.k_{K}.dt_{thresh}/       # Consensus clustering plots
    │       ├── density_filter.png
    │       └── clustering.png
    │
    ├── Inference.gene_spectra_score.k_{K}.dt_{thresh}.txt      # Z-scored gene spectra (genes x programs)
    ├── Inference.gene_spectra_tpm.k_{K}.dt_{thresh}.txt        # TPM-scaled gene spectra
    ├── Inference.usages.k_{K}.dt_{thresh}.consensus.txt        # Consensus usage matrix (cells x programs)
    │
    ├── adata/                                            # Compiled MuData objects
    │   └── cNMF_{K}_{thresh}.h5mu                        # MuData with "rna" + "cNMF" modalities
    │
    ├── loading/                                          # Gene loading matrices
    │   └── cNMF_{K}_{thresh}_loading.npz                  # Sparse gene x program loading matrix
    │
    ├── prog_data/                                        # Program-level summary data
    │   └── cNMF_{K}_{thresh}_prog_data.csv                # Per-program metadata (top genes, stats)
    │
    ├── diagnosis_plots/                                  # Diagnostic visualizations
    │   ├── k_selection.png                                # K-selection plot (stability vs error)
    │   ├── clustering_k{K}_dt{thresh}.png                 # Consensus clustering plots
    │   ├── elbow_curves_k{K}_dt{thresh}.pdf               # Gene loading elbow curves per program
    │   ├── usage_heatmap_k{K}_dt{thresh}.pdf              # Program usage heatmap (cells x programs)
    │   └── loading_violins_k{K}_dt{thresh}.pdf            # Gene loading distribution violins
    │
    └── Annotation/                                       # Gene annotation Excel files
        └── {K}_{thresh}.xlsx                              # Top genes per program with Enrichr annotations
```

### Key Files

| File | Description | Used by |
|------|-------------|---------|
| `adata/cNMF_{K}_{thresh}.h5mu` | Primary output — MuData with programs + expression | Evaluation, Interpretation, Calibration |
| `cnmf_tmp/Inference.norm_counts.h5ad` | Normalized counts (HVG-filtered) | Evaluation (explained variance) |
| `cnmf_tmp/Inference.k_selection_stats.df.npz` | Stability/error per K | K-selection plotting |
| `Inference.gene_spectra_score.k_{K}.dt_{thresh}.txt` | Z-scored gene loadings (genes x programs) | Gene-level analysis, enrichment |
| `Inference.usages.k_{K}.dt_{thresh}.consensus.txt` | Cell x program usage matrix | Downstream visualization |
| `diagnosis_plots/` | Diagnostic plots (elbow, heatmap, violin) | Visual QC of inference results |
| `Annotation/{K}_{thresh}.xlsx` | Gene annotations per program | Manual inspection |

### Notes

- `{K}` = number of components (e.g., 30, 50, 100). `{thresh}` = density threshold with `.` replaced by `_` (e.g., `0_4`, `2_0`).
- Files in `cnmf_tmp/` use the prefix `Inference.` (the cNMF internal run name) rather than the user-specified run name.
- The `adata/*.h5mu` files are the primary input to the downstream **Evaluation** pipeline.
- When running via SLURM with array jobs (one K per job), each K produces its own `{run_name}_{K}/Inference/` directory.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc

# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from torch_cnmf import cNMF

from Stage1_Inference.src import (
    run_cnmf_consensus, get_top_indices_fast, annotate_genes_to_excel,
    rename_and_move_files_NMF, rename_all_NMF, compile_results
)
from Stage1_Inference.src.plot_diagnostics import generate_all_plots

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch

# Check if CUDA is available
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# Get current GPU details
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB")

## Step 1. Set Parameters

In [ ]:
# cNMF parameters
n_iter = 10
num_highvar_genes = 5451  
components = [30, 50, 60, 80, 100, 200, 250, 300]
sel_threshs = [0.4, 0.8, 2.0]
seed = 14
beta_loss = 'frobenius'
init = "random"
mode = "batch"
algo = 'halsvar'
tol = 1e-7
batch_max_epoch = 1000
minibatch_max_epoch = 200
minibatch_size = 100000
minibatch_max_iter = 1000
minibatch_usage_tol = 0.005
minibatch_spectra_tol = 0.005
use_gpu = True
batch_hals_max_iter = 1000
batch_hals_tol = 0.005

# annotation parameters
sk_cd_refit = True
gene_num = 300
species = "human"
gene_names_key = "symbol"  # column in adata.var with gene names; set to None to use var_names
 
# keys 
categorical_key="sex"
guide_assignment_key = "guide_assignment"
guide_names_key = "guide_names"
guide_targets_key = "guide_targets" 
Ensembl_start = 'ENSG'
Gene_symbol_key = 'symbol'

# IO
counts_fn = "/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Data/IGVF_D0_example.h5ad"
output_directory = "/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Results"
run_name = "111025_D0_IGVF_10iter_torch_halsvar_batch_e7_v100s_test"

# resource files
nmf_seeds_path = None

## Step 2. Load Seeds

In [ ]:
if nmf_seeds_path is not None:
    nmf_seeds = np.load(nmf_seeds_path)
else:
    nmf_seeds = None

## Step 3. Load Data

In [ ]:
adata = sc.read(counts_fn)

## Step 4. Filter Genes

In [ ]:
mask = ~adata.var[Gene_symbol_key].str.startswith(Ensembl_start)
adata = adata[:, mask].copy()

run_dir = f'{output_directory}/{run_name}'
inference_dir = f'{run_dir}/Inference'
os.makedirs(inference_dir, exist_ok=True)

filtered_path = f'{inference_dir}/adata_without_noncoding.h5ad'
adata.write(filtered_path)
counts_fn = filtered_path

## Step 5. Prepare

In [ ]:
run_dir = f'{output_directory}/{run_name}'
cnmf_obj = cNMF(output_dir=run_dir, name='Inference')

In [ ]:
cnmf_obj.prepare(counts_fn=counts_fn, components=components, n_iter=n_iter, densify=False, tpm_fn=None, num_highvar_genes=num_highvar_genes, genes_file=None, beta_loss=beta_loss, 
                algo=algo, mode=mode, tol=tol, n_jobs=-1, init="random",
                seed=seed, use_gpu=use_gpu, 
                alpha_usage=0.0, alpha_spectra=0.0, 
                l1_ratio_usage=0.0, l1_ratio_spectra=0.0,
                minibatch_usage_tol=minibatch_usage_tol, minibatch_spectra_tol=minibatch_spectra_tol,
                fp_precision='float', 
                batch_max_epoch=batch_max_epoch, batch_hals_tol=batch_hals_tol, batch_hals_max_iter=batch_hals_max_iter,
                minibatch_max_epoch=minibatch_max_epoch, minibatch_size=minibatch_size, minibatch_max_iter=minibatch_max_iter,
                sk_cd_refit=sk_cd_refit, nmf_seeds=nmf_seeds)

## Step 6. Factorize

In [ ]:
cnmf_obj.factorize(skip_completed_runs=True)

## Step 7. Combine

In [ ]:
cnmf_obj.combine()

## Step 8. K Selection

In [ ]:
cnmf_obj.k_selection_plot()

## Step 9. Consensus

In [ ]:
run_cnmf_consensus(cnmf_obj, 
                   components=components, 
                   density_thresholds=sel_threshs)

## Step 10. Save Results to MuData

In [ ]:
compile_results(run_dir, 'Inference', components=components, sel_threshs=sel_threshs,
     guide_names_key=guide_names_key, guide_targets_key=guide_targets_key, categorical_key=categorical_key, 
     guide_assignment_key=guide_assignment_key, gene_names_key=gene_names_key)

## Step 11. Gene Annotation

In [ ]:
inference_dir = f'{output_directory}/{run_name}/Inference'
os.makedirs(f'{inference_dir}/Annotation', exist_ok=True)
for i in sel_threshs:
    for k in components:
        df = pd.read_csv('{inference_dir}/Inference.gene_spectra_score.k_{k}.dt_{sel_thresh}.txt'.format(
                                                                                inference_dir=inference_dir,
                                                                                k=k,
                                                                                sel_thresh = str(i).replace('.','_')),
                                                                                sep='\t', index_col=0)   
        overlap = get_top_indices_fast(df, gene_num=300)
        annotate_genes_to_excel(overlap, species = 'human', output_file = f'{inference_dir}/Annotation/{k}_{i}.xlsx')

## Step 12. Diagnostic Plots

In [ ]:
run_dir = f'{output_directory}/{run_name}'
plots_dir = generate_all_plots(
    run_dir=run_dir,
    run_name='Inference',
    K_list=components,
    sel_thresh_list=sel_threshs,
    categorical_key=categorical_key,
)
print(f"Diagnostic plots saved to: {plots_dir}")